# Case study: loading the Berlin listings with keys and constraints

*Session 3, extras · Author: course team · Licence: CC BY 4.0*

**Why.** The course database `airbnb` was built quickly by `make_airbnb_postgres.py`: every table has a primary key,
but no foreign keys and no rules for the values. A team database should be stricter: loaded the same way every time
and protected by rules that every member's code must respect. This notebook builds a second, strict database on the
same local PostgreSQL server from the same Parquet files, schema first, and documents every decision the rules force
us to make. It is the long version of the section "Creating tables and loading data" of the block 1 lecture, and a
model for the team project's loader.

A reproducible **loading script** always does the same steps in the same order:

1. create the tables with their keys and constraints ([`../sql/01-schema.sql`](../sql/01-schema.sql));
2. load the parent table (`listings`) before the child tables (`calendar`, `reviews`), because PostgreSQL checks
   every foreign key when a row is written;
3. load each table in one transaction, so that a failed load leaves no half-loaded table;
4. check the counts against the source files and write down every row that was changed or left out.

The strict database `airbnb_strict` is created at the start and deleted at the end; nothing is written into the
repository.

In [1]:
import io
from pathlib import Path

import pandas as pd
import pixeltable_pgserver as pgserver
import psycopg

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data" / "airbnb"
assert (DATA / "listings.parquet").exists(), "download airbnb-v1.zip from the course drive into case-study/data/ (see case-study/README.md)"
SCHEMA = (ROOT / "sessions" / "03-sql" / "workbooks" / "sql" / "01-schema.sql").read_text()

server = pgserver.get_server(DATA / "pgdata")      # the local server of the course (block 1, Setup)
admin = psycopg.connect(server.get_uri(), autocommit=True)       # the default database "postgres"
admin.execute("DROP DATABASE IF EXISTS airbnb_strict")
admin.execute("CREATE DATABASE airbnb_strict")                  # a second, empty database on the same server
print("empty database airbnb_strict created")

empty database airbnb_strict created


## 1. The schema

Read the schema first: which columns are keys, which rules are checks? The date columns have the type `DATE`, so
PostgreSQL itself rejects text that is not a valid date, such as `'2009-13-01'`; no `CHECK` is needed for that. A
`CHECK` is still needed for rules that the type cannot express, such as "the first day of a month" for `reviews.month`.

> **Note.** A `CHECK` rejects a row only when its condition is *false*; an *unknown* (NULL) result passes. So
> `CHECK (price > 0)` accepts a missing price. Add `NOT NULL` where a value must be present.

In [2]:
print(SCHEMA)

-- Schema of a strict course database for the Berlin Airbnb case study, with keys and constraints
-- (PostgreSQL). Used by extras/02-loading-with-constraints.ipynb: run it on an empty database, then
-- load the parent table (listings) before the child tables (calendar, reviews). PostgreSQL always
-- checks foreign keys. Dates are of type DATE, so an impossible date such as '2009-13-01' is rejected
-- by the type itself; available is a BOOLEAN.
-- Author: course team, licence CC BY 4.0

DROP TABLE IF EXISTS calendar;
DROP TABLE IF EXISTS reviews;
DROP TABLE IF EXISTS listings;

CREATE TABLE listings (                                   -- one row per listing of the snapshot
    id                   BIGINT PRIMARY KEY,              -- some ids have 19 digits
    host_id              BIGINT NOT NULL,
    district             TEXT NOT NULL,
    neighbourhood        TEXT NOT NULL,
    latitude             DOUBLE PRECISION NOT NULL CHECK (latitude BETWEEN 52.33 AND 52.68),
    longitude      

## 2. A first attempt, exactly as the files are

The loader creates the tables and tries to load the listings without any change, with `COPY` (block 1). `with
con.transaction():` makes the load one transaction: if one row fails, no row is stored. pandas writes missing values as
empty fields, which `COPY` reads as NULL.

In [3]:
LISTING_COLS = ["id", "host_id", "district", "neighbourhood", "latitude", "longitude", "room_type", "accommodates",
                "bedrooms", "price", "minimum_nights", "maximum_nights", "availability_365", "number_of_reviews",
                "first_review", "last_review", "review_scores_rating", "license_status"]
WHOLE_NUMBERS = ["bedrooms", "minimum_nights", "maximum_nights"]   # float in Parquet because of missing values
SENTINEL = 2_147_483_647          # the largest 32-bit integer


def fresh_database() -> psycopg.Connection:
    """Connect to airbnb_strict and create the tables of the schema (dropping old ones)."""
    con = psycopg.connect(server.get_uri("airbnb_strict"), autocommit=True)
    con.execute(SCHEMA)
    return con


def load(con: psycopg.Connection, table: str, frame: pd.DataFrame) -> None:
    """Load all rows of a DataFrame with COPY, in one transaction (an empty field becomes NULL)."""
    buffer = io.StringIO()
    frame.to_csv(buffer, index=False, header=False, date_format="%Y-%m-%d")
    with con.transaction(), con.cursor() as cur:
        with cur.copy(f"COPY {table} ({', '.join(frame.columns)}) FROM STDIN (FORMAT csv)") as copy:
            copy.write(buffer.getvalue())


listings = pd.read_parquet(DATA / "listings.parquet", columns=LISTING_COLS)
listings[WHOLE_NUMBERS] = listings[WHOLE_NUMBERS].astype("Int64")

con = fresh_database()
try:
    load(con, "listings", listings)
except psycopg.errors.CheckViolation as error:
    print("rejected:", error.diag.message_primary)
print("listings stored:", con.execute("SELECT COUNT(*) FROM listings").fetchone()[0])

rejected: new row for relation "listings" violates check constraint "listings_maximum_nights_check"
listings stored: 0


The database refuses: the message names the constraint `listings_maximum_nights_check`. Two listings carry the
maximum stay 2,147,483,647, the largest 32-bit integer, which is a software default and not a stay (Session 4 finds
this placeholder too). Because the load was one transaction, not a single listing was stored.

## 3. Decisions, then the load

Each change to the data is a **decision**, and each decision is written down:

1. `maximum_nights = 2,147,483,647` becomes NULL (2 listings).
2. Calendar and review rows of listings that are not in the listings table are not loaded, but counted: the foreign
   key would reject them, and Inside Airbnb scraped the files separately, so these listings went offline between two
   scrapes (block 1).

In [4]:
decisions = []
con.close()
con = fresh_database()

n_sentinel = int(listings["maximum_nights"].eq(SENTINEL).sum())
listings["maximum_nights"] = listings["maximum_nights"].where(listings["maximum_nights"] != SENTINEL)
decisions.append(("listings", "maximum_nights = 2,147,483,647 set to NULL", n_sentinel))
load(con, "listings", listings)

calendar = pd.read_parquet(DATA / "calendar.parquet", columns=["listing_id", "date", "available"])
try:                                                    # first try: all rows
    load(con, "calendar", calendar)
except psycopg.errors.ForeignKeyViolation as error:
    print("calendar rejected:", error.diag.message_primary)
    print(error.diag.message_detail)

reviews = pd.read_parquet(DATA / "reviews_monthly.parquet")

known = set(listings["id"])
for table, frame in [("calendar", calendar), ("reviews", reviews)]:
    keep = frame["listing_id"].isin(known)
    decisions.append((table, "rows of listings not in listings (not loaded)", int((~keep).sum())))
    load(con, table, frame[keep])

pd.DataFrame(decisions, columns=["table", "decision", "rows"])

calendar rejected: insert or update on table "calendar" violates foreign key constraint "calendar_listing_id_fkey"
Key (listing_id)=(1025263460868613348) is not present in table "listings".


,table,decision,rows
0,listings,"maximum_nights = 2,147,483,647 set to NULL",2
1,calendar,rows of listings not in listings (not loaded),28835
2,reviews,rows of listings not in listings (not loaded),499


In [5]:
# check the counts against the source files
with con.cursor() as cur:
    cur.execute("""
        SELECT 'listings' AS table_name, COUNT(*) AS n FROM listings
        UNION ALL SELECT 'calendar', COUNT(*) FROM calendar
        UNION ALL SELECT 'reviews', COUNT(*) FROM reviews
    """)
    counts = pd.DataFrame(cur.fetchall(), columns=["table_name", "n"])
counts["in_file"] = [len(listings), len(calendar), len(reviews)]
counts.assign(not_loaded=lambda d: d["in_file"] - d["n"])

,table_name,n,in_file,not_loaded
0,listings,10221,10221,0
1,calendar,3730665,3759500,28835
2,reviews,180157,180656,499


The differences are exactly the rows we decided not to load, and we know which listings they belong to.

## 4. The constraints at work

Try to insert invalid rows. The database rejects them, and the name of the error class says which kind of rule
failed. A value of the wrong type, such as an impossible date, is rejected by the type before any constraint is
checked (`InvalidDatetimeFormat` or `DatetimeFieldOverflow`).

In [6]:
some_id = int(listings["id"].iloc[0])
COLS = ("id, host_id, district, neighbourhood, latitude, longitude, room_type, accommodates, availability_365, "
        "number_of_reviews, license_status")
bad_rows = {
    "unknown room type": f"INSERT INTO listings ({COLS}) VALUES "
                         "(1, 1, 'Mitte', 'Alexanderplatz', 52.52, 13.41, 'Castle', 2, 0, 0, 'missing')",
    "outside Berlin": f"INSERT INTO listings ({COLS}) VALUES "
                      "(2, 1, 'Mitte', 'Alexanderplatz', 48.14, 11.58, 'Private room', 2, 0, 0, 'missing')",
    "unknown listing": "INSERT INTO calendar VALUES (3, '2026-07-01', TRUE)",
    "same night twice": f"INSERT INTO calendar SELECT listing_id, date, available FROM calendar "
                        f"WHERE listing_id = {some_id} LIMIT 1",
    "not a date": f"INSERT INTO reviews VALUES ({some_id}, '2009-13-01', 1)",
    "not a first of month": f"INSERT INTO reviews VALUES ({some_id}, '2009-01-15', 1)",
    "zero reviews row": f"INSERT INTO reviews VALUES ({some_id}, '2009-01-01', 0)",
}
for name, statement in bad_rows.items():
    try:
        con.execute(statement)
        print(f"{name:20s} accepted (!)")
    except psycopg.Error as error:
        print(f"{name:20s} rejected ({type(error).__name__}): {error.diag.message_primary[:80]}")

unknown room type    rejected (CheckViolation): new row for relation "listings" violates check constraint "listings_room_type_ch
outside Berlin       rejected (CheckViolation): new row for relation "listings" violates check constraint "listings_latitude_che
unknown listing      rejected (ForeignKeyViolation): insert or update on table "calendar" violates foreign key constraint "calendar_l
same night twice     rejected (UniqueViolation): duplicate key value violates unique constraint "calendar_pkey"
not a date           rejected (DatetimeFieldOverflow): date/time field value out of range: "2009-13-01"
not a first of month rejected (CheckViolation): new row for relation "reviews" violates check constraint "reviews_month_check"
zero reviews row     rejected (CheckViolation): new row for relation "reviews" violates check constraint "reviews_n_reviews_chec


## 5. What the database contains

A few facts computed from the database, together with the loading decisions. A team keeps such a summary next to its
project data, with the source, the licence and the known limitations, so that a new reader can judge what the data
can and cannot answer. A comparison is a `boolean` in PostgreSQL; `::int` turns it into 1 or 0, so that its average
is a share.

In [7]:
with con.cursor() as cur:
    cur.execute("""
        SELECT COUNT(*) AS n_listings,
               COUNT(DISTINCT host_id) AS n_hosts,
               COUNT(DISTINCT district) AS n_districts,
               COUNT(DISTINCT neighbourhood) AS n_neighbourhoods,
               ROUND(AVG((price IS NULL)::int), 3) AS share_price_missing,
               ROUND(AVG((minimum_nights >= 28)::int), 3) AS share_medium_term,
               ROUND(AVG((number_of_reviews = 0)::int), 3) AS share_without_review,
               ROUND(AVG((license_status = 'registration number')::int), 3) AS share_registered,
               (SELECT MIN(month) FROM reviews) AS first_review_month,
               (SELECT MIN(date) FROM calendar) AS calendar_from,
               (SELECT MAX(date) FROM calendar) AS calendar_until
        FROM listings
    """)
    facts = pd.DataFrame(cur.fetchall(), columns=[column.name for column in cur.description]).T.rename(columns={0: "value"})
for table, decision, rows in decisions:
    facts.loc[f"{table}: {decision}"] = rows
con.close()
admin.execute("DROP DATABASE airbnb_strict")            # remove the strict database again
admin.close()
facts

,value
n_listings,10221
n_hosts,6660
n_districts,12
n_neighbourhoods,137
share_price_missing,0.339
share_medium_term,0.351
share_without_review,0.196
share_registered,0.352
first_review_month,2009-06-01
calendar_from,2026-06-26


**Discussion.** Which limitations should a reader of an analysis based on these tables know? For example:

- the reviews are those of listings still online in June 2026: deleted listings took their reviews with them
  (**survivorship**), so the growth since 2015 overstates the growth of the market;
- a night that is not available is booked *or* blocked: the calendar does not measure demand;
- prices are asking prices for the next free night, a third of the listings show none, and medium-term listings
  have a different kind of price;
- locations are shifted by up to about 150 m by Airbnb to protect the hosts (Inside Airbnb, *Data assumptions*).

> **Important.** Inside Airbnb publishes its data under the Creative Commons Attribution 4.0 licence (CC BY 4.0):
> reuse is allowed with attribution. The data are scraped from public listing pages and contain personal data, which
> the lecturer's preparation script removed before the course download (host names, review texts, names typed into the registration field).
> Document both the licence and these steps with the data, and report results in aggregate, never naming hosts.